# Pipeline v14 (v4 data) -- CoT LoRA Fine-tune
**Retrain tren data da fix: Logic_Based_Educational_Queries_final_v4.json (318 records)**

Khac ban truoc: DATASET_PATH -> train v4 (318 records, ~636 questions).
Output LoRA: `/kaggle/working/qwen3_cot_lora_v14_v4` -- upload lam dataset de v13.5/v13.6/submission dung.

Split SEED=42, TRAIN_RATIO=0.80 (giu nguyen de val so sanh duoc).
Sau khi chay: val accuracy in ra cuoi notebook.


In [ ]:
#!/usr/bin/env python3
"""
notebook_v14_cot_lora_finetune.py
EXACT 2026 -- v14: Train Qwen3-8B LoRA on dataset's `explanation` field
to produce pedagogical CoT reasoning + Final Answer marker.
Output: /kaggle/working/qwen3_cot_lora_v14 (PEFT adapter for vLLM).
"""


In [ ]:
# ================= Kaggle T4 fix =================
import os, shutil, glob
STUB_DIR = "/tmp/cuda_stubs"; os.makedirs(STUB_DIR, exist_ok=True)
stub = os.path.join(STUB_DIR, "libcuda.so")
if os.path.exists(stub) or os.path.islink(stub): os.remove(stub)
for c in ["/usr/lib/x86_64-linux-gnu/libcuda.so.1", "/usr/lib/x86_64-linux-gnu/libcuda.so",
          *glob.glob("/usr/**/libcuda.so*", recursive=True)]:
    if os.path.exists(c) and not os.path.islink(c):
        os.symlink(c, stub); break
os.environ["LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LD_LIBRARY_PATH", "")
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)
print("Kaggle T4 fixes applied")


In [ ]:
# ================= INSTALL =================
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages",
                "unsloth", "trl<=0.24.0", "datasets<4.4.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--break-system-packages",
                "--no-cache-dir", "peft", "accelerate", "bitsandbytes"], check=True)
import torch
print("PyTorch", torch.__version__, "CUDA", torch.cuda.is_available())


In [ ]:
# ================= CONFIG =================
MODEL_PATH       = "/kaggle/input/models/qwen-lm/qwen-3/transformers/8b/1"
# ---- Robust Kaggle path resolver (handles mount-path variations) ----
import os as _os
def _resolve(cands, label="path"):
    for p in cands:
        if _os.path.exists(p):
            print(f"  resolved {label}: {p}")
            return p
    print(f"  WARNING {label}: none of candidates exist; using first: {cands[0]}")
    return cands[0]

TRAIN_PATH = _resolve([
    "/kaggle/input/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries_final_v4.json",
], "TRAIN")
TEST_PATH = _resolve([
    "/kaggle/input/test-pipeline/generated_v4style_300.json",
    "/kaggle/input/datasets/nguyenminhtric/test-pipeline/generated_v4style_300.json",
], "TEST")
DATASET_PATH = TRAIN_PATH    # eval notebooks use the labeled train file

# DATASET_PATH set by resolver below
LORA_OUTPUT_DIR  = "/kaggle/working/qwen3_cot_lora_v14_v4"  # NEW: trained on v4 data
CHECKPOINT_DIR   = "/kaggle/working/cot_lora_ckpt"
EVAL_OUTPUT_PATH = "/kaggle/working/v14_eval_results.json"

# Split (MUST match v13: CAL_SEED=42, CAL_TRAIN_RATIO=0.80, sample-level)
SEED          = 42
TRAIN_RATIO   = 0.80

MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT   = True

LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0
TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

EPOCHS         = 2
BATCH_SIZE     = 2
GRAD_ACCUM     = 4
LEARNING_RATE  = 1e-4
WARMUP_STEPS   = 10
WEIGHT_DECAY   = 0.01
OPTIMIZER      = "adamw_8bit"
LR_SCHEDULER   = "linear"
LOGGING_STEPS  = 10

# Must match inference (we use plain CoT, not thinking-mode tokens)
ENABLE_THINKING_TRAIN = False

# Inline eval
EVAL_MAX_NEW_TOKENS = 512
EVAL_TEMPERATURE    = 0.1   # low temp for evaluation
EVAL_N_SAMPLES      = None  # None = all val; set e.g. 30 for quick sanity

# System prompt -- must be identical at training and inference
SYSTEM_PROMPT = (
    "You are a careful logic tutor. Given a list of premises and a question, "
    "reason step-by-step by referencing specific premises (e.g. 'From premise 3...'). "
    "Then state your conclusion on a final line in the exact format: 'Final Answer: X' "
    "where X is one of: Yes, No, Unknown, A, B, C, or D."
)
print("Config loaded")


In [ ]:
# ================= LOAD MODEL + ATTACH LoRA =================
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    target_modules=TARGET_MODULES, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED,
)
trn = sum(p.numel() for p in model.parameters() if p.requires_grad)
tot = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trn:,}/{tot:,} ({100*trn/tot:.2f}%)")


In [ ]:
# ================= BUILD CoT DATASET =================
import json, random, re
from datasets import Dataset

with open(DATASET_PATH, encoding="utf-8") as f:
    raw = json.load(f)
n = len(raw)
print(f"Dataset samples: {n}")

# Sample-level split, deterministic, MATCHES v13.3
rng = random.Random(SEED)
ids = list(range(n)); rng.shuffle(ids)
n_tr = int(n * TRAIN_RATIO)
train_ids = set(ids[:n_tr]); val_ids = set(ids[n_tr:])
train_samples = [raw[i] for i in range(n) if i in train_ids]
val_samples   = [raw[i] for i in range(n) if i in val_ids]
print(f"Train: {len(train_samples)} samples | Val: {len(val_samples)} samples")

def build_user_msg(premises_nl, question):
    lines = ["### Premises"]
    for i, p in enumerate(premises_nl):
        lines.append(f"P{i+1}. {p}")
    return "\n".join(lines) + f"\n\n### Question\n{question}"

def build_assistant_msg(explanation, gold):
    # Append "Final Answer: X" unless explanation already ends with such a line
    txt = explanation.strip()
    last = txt.split("\n")[-1].lower()
    if not (re.search(r"\bfinal\s+answer\b", last) or
            re.search(r"\banswer\s*:", last)):
        txt = f"{txt}\n\nFinal Answer: {gold}"
    return txt

# Build records (one per question)
def to_records(samples):
    out = []
    for s in samples:
        nls = s.get("premises-NL", [])
        for q, a, e in zip(s.get("questions", []), s.get("answers", []),
                            s.get("explanation", [])):
            if not e or not str(a).strip():
                continue
            out.append({"conversations": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": build_user_msg(nls, q)},
                {"role": "assistant", "content": build_assistant_msg(e, a)},
            ]})
    return out

train_records = to_records(train_samples)
print(f"Train examples (questions): {len(train_records)}")
print("\n--- Sample training record ---")
print("USER:", train_records[0]["conversations"][1]["content"][:300], "...")
print("\nASSISTANT:", train_records[0]["conversations"][2]["content"][:400])

train_ds = Dataset.from_list(train_records)
print(f"\nDataset: {train_ds}")


In [ ]:
# ================= TRAINER (responses-only) =================
import torch
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

def formatting_func(examples):
    convos = examples["conversations"]
    def _fmt(c):
        try:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False,
                enable_thinking=ENABLE_THINKING_TRAIN)
        except TypeError:
            return tokenizer.apply_chat_template(
                c, tokenize=False, add_generation_prompt=False)
    if len(convos) > 0 and isinstance(convos[0], dict):
        return [_fmt(convos)]
    return [_fmt(c) for c in convos]

sft = SFTConfig(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    optim=OPTIMIZER,
    lr_scheduler_type=LR_SCHEDULER,
    logging_steps=LOGGING_STEPS,
    save_strategy="epoch",
    save_total_limit=1,
    seed=SEED,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)
trainer_obj = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train_ds,
                         args=sft, formatting_func=formatting_func)
trainer_obj = train_on_responses_only(
    trainer_obj,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
print(f"Trainer ready | examples={len(train_ds)} | effective batch={BATCH_SIZE*GRAD_ACCUM}")


In [ ]:
# ================= TRAIN + SAVE LoRA =================
import time, os
t0 = time.time()
result = trainer_obj.train()
dur_min = (time.time() - t0) / 60
print(f"Training done in {dur_min:.1f} min | final loss = {result.training_loss:.4f}")

# Save adapter
model.save_pretrained(LORA_OUTPUT_DIR)
tokenizer.save_pretrained(LORA_OUTPUT_DIR)
print(f"Saved LoRA -> {LORA_OUTPUT_DIR}")
print("Contents:", os.listdir(LORA_OUTPUT_DIR))


In [ ]:
# ================= INLINE EVAL on VAL (vs pure_qwen 33% baseline) =================
import re, json, time
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
print("Model in inference mode")

def extract_final_answer(text):
    if not text: return "UNPARSEABLE"
    # Tier 1: 'Final Answer: X' (case-insensitive)
    m = re.search(r"final\s*answer\s*[:\-]?\s*([A-Da-d]\b|Yes|No|Unknown|YES|NO|UNKNOWN|yes|no|unknown)",
                  text, re.I)
    if m:
        a = m.group(1).strip()
        return a.upper() if a.lower() in ("yes","no","unknown") else a.upper()
    # Tier 2: standalone last line
    for line in reversed(text.strip().split("\n")):
        s = line.strip()
        if s in ("Yes","No","Unknown","A","B","C","D"): return s.upper() if len(s)>1 else s
    # Tier 3: any answer-like token in last 100 chars
    tail = text[-200:]
    for tok in ["Yes","No","Unknown","A","B","C","D"]:
        if re.search(rf"\b{tok}\b", tail):
            return tok.upper() if len(tok)>1 else tok
    return "UNPARSEABLE"

# Build val records (use same code paths as training)
val_records = []
for s in val_samples:
    nls = s.get("premises-NL", [])
    for q_idx, (q, a) in enumerate(zip(s.get("questions", []), s.get("answers", []))):
        val_records.append({"sample_id": s.get("idx", [[]])[0] if False else None,
                            "q_idx": q_idx, "user": build_user_msg(nls, q),
                            "gold": str(a).strip()})

if EVAL_N_SAMPLES is not None:
    val_records = val_records[:EVAL_N_SAMPLES]
print(f"Evaluating {len(val_records)} val questions...")

def gen_one(user_msg):
    msgs = [{"role":"system","content":SYSTEM_PROMPT}, {"role":"user","content":user_msg}]
    try:
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                            return_tensors="pt", enable_thinking=ENABLE_THINKING_TRAIN).to("cuda")
    except TypeError:
        ids = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True,
                                            return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=EVAL_MAX_NEW_TOKENS,
                         temperature=EVAL_TEMPERATURE, do_sample=(EVAL_TEMPERATURE>0),
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)

t0 = time.time(); results = []
correct = 0
for j, rec in enumerate(val_records):
    raw = gen_one(rec["user"])
    pred = extract_final_answer(raw)
    ok = (str(pred).upper() == str(rec["gold"]).upper())
    if ok: correct += 1
    results.append({"q_idx": rec["q_idx"], "gold": rec["gold"], "pred": pred,
                    "correct": ok, "raw_tail": raw[-300:]})
    if (j+1) % 25 == 0:
        print(f"  [{j+1}/{len(val_records)}] running acc = {correct/(j+1):.1%}  ({(time.time()-t0)/60:.1f} min)")

acc = correct / max(len(val_records), 1)
dur = (time.time() - t0) / 60
print("\n" + "="*70)
print(f"  v14 INLINE EVAL on VAL (matches v13.3 val split)")
print("="*70)
print(f"  Questions evaluated : {len(val_records)}")
print(f"  Correct             : {correct}")
print(f"  ACCURACY            : {acc:.1%}")
print(f"  Duration            : {dur:.1f} min")
print("-"*70)
print(f"  Baseline pure_qwen (v13.3 val) : 33.1%")
print(f"  Improvement                    : {acc - 0.331:+.1%}")
print(f"  Success threshold (>=50%)      : {'PASS' if acc >= 0.50 else 'FAIL'}")
print("="*70)

# Save
summary = {
    "meta": {"version": "v14_cot_lora", "seed": SEED, "train_ratio": TRAIN_RATIO,
             "lora_dir": LORA_OUTPUT_DIR, "duration_min": dur,
             "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")},
    "metrics": {"val_accuracy": acc, "correct": correct, "total": len(val_records),
                "baseline_pure_qwen": 0.331, "improvement_pp": acc - 0.331},
    "per_question": results,
}
with open(EVAL_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"\nSaved eval: {EVAL_OUTPUT_PATH}")

print("\nNEXT STEPS:")
print(f"  - LoRA adapter saved at: {LORA_OUTPUT_DIR}")
print( "  - To integrate in v13.3: in that notebook's config cell set")
print(f"      FORMALIZER_LORA_PATH = '{LORA_OUTPUT_DIR}'  (or wire a new flag)")
print( "    and pass use_lora=True to batch_generate for the CoT step.")
print( "  - Re-run v13.3 -> expect new oracle ceiling + better pure_qwen + better aggregators")


In [ ]:
# ================= EXTENDED EVAL -- FULL DATA + TRAIN vs VAL + SAMPLE REVIEW =================
# Muc dich: phat hien overfit bang cach so accuracy tren TRAIN (da hoc) va VAL (chua thay).
# Reuses: gen_one(), extract_final_answer(), build_user_msg(), SYSTEM_PROMPT, train_samples,
#         val_samples, EVAL_MAX_NEW_TOKENS, EVAL_TEMPERATURE, LORA_OUTPUT_DIR (tu cell tren).
# Output:  /kaggle/working/v14_eval_extended.json
# ===========================================================================================
import re, json, time, random as _rnd
from collections import defaultdict, Counter

# ---- Build eval records for BOTH splits ----
# (val_records already in scope from cell 9; rebuild with split tag for unified loop)
def _build_records(samples, split_name):
    out = []
    for s_i, s in enumerate(samples):
        nls = s.get("premises-NL", [])
        fols = s.get("premises-FOL", [])
        for q_idx, (q, gold) in enumerate(zip(s.get("questions", []), s.get("answers", []))):
            out.append({
                "split":      split_name,
                "sample_i":   s_i,
                "q_idx":      q_idx,
                "user":       build_user_msg(nls, q),
                "question":   q,
                "gold":       str(gold).strip(),
                "premises_nl": nls,
            })
    return out

all_eval_records = _build_records(train_samples, "train") + _build_records(val_samples, "val")
train_eval = [r for r in all_eval_records if r["split"] == "train"]
val_eval   = [r for r in all_eval_records if r["split"] == "val"]
print(f"Eval plan: {len(train_eval)} train questions + {len(val_eval)} val questions "
      f"= {len(all_eval_records)} total")

# ---- Inference loop (greedy, same settings as cell 9) ----
print(f"\nRunning inference (temperature={EVAL_TEMPERATURE}, max_new_tokens={EVAL_MAX_NEW_TOKENS})...")
print("-" * 70)
t0 = time.time()
correct_so_far = 0
for j, rec in enumerate(all_eval_records):
    raw  = gen_one(rec["user"])
    pred = extract_final_answer(raw)
    ok   = (str(pred).upper() == str(rec["gold"]).upper())
    rec["pred"]     = pred
    rec["raw_full"] = raw
    rec["correct"]  = ok
    if ok: correct_so_far += 1
    if (j + 1) % 30 == 0 or (j + 1) == len(all_eval_records):
        elapsed = (time.time() - t0) / 60
        split_tag = rec["split"]
        print(f"  [{j+1:4d}/{len(all_eval_records)}]  running={correct_so_far/(j+1):.1%}"
              f"  split={split_tag}  elapsed={elapsed:.1f}min")

total_dur = (time.time() - t0) / 60

# ---- Partition results ----
train_r = [r for r in all_eval_records if r["split"] == "train"]
val_r   = [r for r in all_eval_records if r["split"] == "val"]

def _acc(lst):
    if not lst: return 0.0, 0, 0
    c = sum(r["correct"] for r in lst)
    return 100 * c / len(lst), c, len(lst)

full_acc, full_c, full_n   = _acc(all_eval_records)
train_acc, train_c, train_n = _acc(train_r)
val_acc, val_c, val_n       = _acc(val_r)
gap = train_acc - val_acc

# ================================================================
# SUMMARY TABLE
# ================================================================
print("\n" + "=" * 70)
print(f"  v14 LoRA -- FULL EVALUATION (train={train_n}q  val={val_n}q)")
print("=" * 70)
print(f"  {'Split':<20} {'Correct':>10} {'Total':>8} {'Accuracy':>10}")
print(f"  {'-'*50}")
print(f"  {'FULL (train+val)':<20} {full_c:>10} {full_n:>8} {full_acc:>9.1f}%")
print(f"  {'TRAIN (seen)':<20} {train_c:>10} {train_n:>8} {train_acc:>9.1f}%")
print(f"  {'VAL (held-out)':<20} {val_c:>10} {val_n:>8} {val_acc:>9.1f}%")
print(f"  {'-'*50}")
print(f"  {'GAP (train - val)':<20} {'':>10} {'':>8} {gap:>+9.1f}pp")
print("=" * 70)

# ---- Overfit diagnosis ----
print(f"\n  Overfit diagnosis (GAP = {gap:+.1f}pp):")
if   gap > 20: print("  -> OVERFIT NANG: model nhớ gần như toàn bộ training data.")
elif gap > 12: print("  -> Overfit đáng kể: train accuracy ảo, val là số thật.")
elif gap > 6:  print("  -> Overfit vừa: bình thường với dataset nhỏ (~500 samples).")
elif gap > 2:  print("  -> Overfit nhẹ: chấp nhận được.")
else:          print("  -> OK: model generalize tốt, train và val nhất quán.")
print(f"  Baseline pure_qwen (v13.3 val)   : 33.1%")
print(f"  v14 val accuracy                  : {val_acc:.1f}%")
print(f"  Improvement vs baseline           : {val_acc - 33.1:+.1f}pp")
print(f"  Success threshold (>=50% val)     : {'PASS' if val_acc >= 50 else 'FAIL'}")
print(f"  Total inference time              : {total_dur:.1f} min")

# ================================================================
# PER-CLASS ACCURACY -- TRAIN vs VAL
# ================================================================
print("\n" + "=" * 70)
print("  Per-class accuracy (TRAIN vs VAL)")
print("=" * 70)
all_labels = sorted(set(r["gold"] for r in all_eval_records))
print(f"  {'Label':>12} {'Train acc':>12} {'Val acc':>12} {'Train n':>9} {'Val n':>8}")
print(f"  {'-'*56}")
for lbl in all_labels:
    tr = [r for r in train_r if r["gold"].upper() == lbl.upper()]
    va = [r for r in val_r   if r["gold"].upper() == lbl.upper()]
    ta, tc, tn = _acc(tr); va_, vc, vn = _acc(va)
    gap_lbl = ta - va_
    flag = ""
    if lbl.lower() in ("no",) and vn and va_ < 60:
        flag = "  <- underfit"
    elif abs(gap_lbl) > 15:
        flag = "  <- overfit lớp này"
    print(f"  {lbl:>12} {ta:>11.0f}% {va_:>11.0f}% {tn:>9} {vn:>8}{flag}")

# ================================================================
# CONFUSION MATRIX -- VAL ONLY (where overfit matters)
# ================================================================
print("\n" + "=" * 70)
print("  Confusion matrix on VAL (expected -> predicted)")
print("=" * 70)
confusion = defaultdict(Counter)
for r in val_r:
    confusion[r["gold"]][r["pred"]] += 1
pred_labels = sorted(set(r["pred"] for r in val_r))
header = f"  {'Expected':>12} | " + " ".join(f"{p:>12}" for p in pred_labels)
print(header)
print("  " + "-" * (14 + 13 * len(pred_labels)))
for gold_lbl in all_labels:
    row = confusion.get(gold_lbl, {})
    total_row = sum(row.values())
    cells = " ".join(f"{row.get(p, 0):>12}" for p in pred_labels)
    acc_lbl = 100 * row.get(gold_lbl, 0) / total_row if total_row else 0
    print(f"  {gold_lbl:>12} | {cells}    n={total_row} acc={acc_lbl:.0f}%")

# ================================================================
# ERROR ANALYSIS -- top wrong patterns on VAL
# ================================================================
wrong_val = [r for r in val_r if not r["correct"]]
print(f"\n  Error analysis on VAL: {len(wrong_val)}/{val_n} wrong ({100*len(wrong_val)/val_n:.1f}%)")
print(f"  {'-'*50}")
wrong_patterns = Counter((r["gold"], r["pred"]) for r in wrong_val)
for (gold, pred), cnt in wrong_patterns.most_common(8):
    print(f"  gold={gold:>8s} -> pred={pred:<8s}  x{cnt}")

# ================================================================
# SAMPLE REVIEW -- 5 TRAIN + 5 VAL with full CoT output
# ================================================================
_rnd.seed(42)

def _print_samples(title, sample_list, n=5, show_cot_lines=6):
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)
    chosen = _rnd.sample(sample_list, min(n, len(sample_list)))
    for i, r in enumerate(chosen, 1):
        status = "CORRECT" if r["correct"] else "WRONG  "
        print(f"\n  [{i}] {status}  gold={r['gold']:>8s}  pred={r['pred']}")
        # Question
        print(f"       Q : {r['question'][:100]}")
        # Show top N premises
        if r["premises_nl"]:
            print(f"       P1: {r['premises_nl'][0][:80]}")
            if len(r["premises_nl"]) > 1:
                print(f"       P2: {r['premises_nl'][1][:80]}")
        # CoT output (first few lines)
        cot_lines = [l for l in r["raw_full"].strip().split("\n") if l.strip()]
        print(f"       CoT ({len(cot_lines)} lines):")
        for line in cot_lines[:show_cot_lines]:
            print(f"         {line[:100]}")
        if len(cot_lines) > show_cot_lines:
            print(f"         ... ({len(cot_lines) - show_cot_lines} more lines)")
        if not r["correct"]:
            # Find where it went wrong
            final_lines = [l for l in cot_lines if "final" in l.lower() or "answer" in l.lower()]
            if final_lines:
                print(f"       !! Final line: {final_lines[-1][:100]}")
            print(f"       !! Expected '{r['gold']}', extracted '{r['pred']}'")

_print_samples("5 mau TRAIN -- da hoc (train acc khong tin duoc neu qua cao)", train_r)
_print_samples("5 mau VAL   -- chua thay (so nay moi la so thuc)", val_r)

# ---- Wrong val cases detailed view ----
if wrong_val:
    print("\n" + "=" * 70)
    print(f"  3 mau SAI tren VAL (chi tiet CoT)")
    print("=" * 70)
    for i, r in enumerate(_rnd.sample(wrong_val, min(3, len(wrong_val))), 1):
        print(f"\n  [{i}] gold={r['gold']:>8s}  pred={r['pred']}")
        print(f"       Q: {r['question'][:110]}")
        cot = r["raw_full"].strip()
        print(f"       CoT ({len(cot.split(chr(10)))} lines, last 400 chars):")
        print(f"       {cot[-400:]}")

# ================================================================
# SAVE
# ================================================================
extended_out = "/kaggle/working/v14_eval_extended.json"
json.dump({
    "meta": {
        "version":        "v14_cot_lora_extended_eval",
        "lora_dir":       LORA_OUTPUT_DIR,
        "seed":           SEED,
        "train_ratio":    TRAIN_RATIO,
        "eval_temp":      EVAL_TEMPERATURE,
        "eval_max_tokens":EVAL_MAX_NEW_TOKENS,
        "duration_min":   total_dur,
        "timestamp":      time.strftime("%Y-%m-%d %H:%M:%S"),
    },
    "summary": {
        "full_acc":   full_acc,  "full_n":   full_n,
        "train_acc":  train_acc, "train_n":  train_n,
        "val_acc":    val_acc,   "val_n":    val_n,
        "gap_pp":     gap,
        "baseline_pure_qwen": 33.1,
        "improvement_vs_baseline_pp": val_acc - 33.1,
    },
    "per_class_val": {
        lbl: {"acc": _acc([r for r in val_r if r["gold"].upper()==lbl.upper()])[0],
              "n":   len([r for r in val_r if r["gold"].upper()==lbl.upper()])}
        for lbl in all_labels
    },
    "confusion_val": {gold: dict(preds) for gold, preds in confusion.items()},
    "n_wrong_val":   len(wrong_val),
    "per_question":  [{k: v for k, v in r.items() if k != "raw_full"}
                      for r in all_eval_records],  # omit raw_full to save space
}, open(extended_out, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(f"\nSaved: {extended_out}")

print("\nNEXT STEPS:")
print(f"  - Val accuracy = {val_acc:.1f}%  (so sanh voi v14match_fixed de biet config nao tot hon)")
print(f"  - GAP train-val = {gap:+.1f}pp  (neu > 10pp thi nen giam epochs hoac lora_r)")
print(f"  - Chay v13.6 voi LoRA nay: dat {val_acc:.1f}% pure_qwen + Z3 calibrator se toi uu hon nua")
print(f"  - LoRA adapter: {LORA_OUTPUT_DIR}")
